PROCESSING RAW DATA

In [3]:
import os
import glob
import pandas as pd

# Process and split the data into 2 datasets: one for each hospital.
def process_and_save_hospital(folder_name, output_filename):
    print(f"Processing {folder_name}...")

    # Find all .psv files in the folder
    search_path = os.path.join('../data/raw/physionet.org/files/challenge-2019/1.0.0/training/', folder_name, '*.psv')
    all_files = glob.glob(search_path)

    # Loop through each file, add ID/Time columns, and concatenate them into a single DataFrame
    df_list = []

    for file in all_files:
        # Read the .psv file
        df = pd.read_csv(file, sep='|')

        #Extract the patient ID from the filename
        df['Patient_ID'] = os.path.basename(file).split('.')[0]

        # Add a Time column (it is specified that each line represents an hour of data)
        df['ICU_Hour'] = range(1, len(df) + 1)

        df_list.append(df)

    # Combine the dataframes into one and save it into a parquet file
    full_df = pd.concat(df_list, ignore_index=True)
    out_path = os.path.join(f'../data/processed/{output_filename}.parquet')
    full_df.to_parquet(out_path, index=False)

    print(f"Saved {output_filename}! Shape: {full_df.shape}\n")

process_and_save_hospital('training_setA', 'train_hospital_A')
process_and_save_hospital('training_setB', 'test_hospital_B')

Processing training_setA...
Saved train_hospital_A! Shape: (790215, 43)

Processing training_setB...
Saved test_hospital_B! Shape: (761995, 43)



Now that we split the raw data into train_hospital_A and test_hospital_B we will start DATA CLEANING (FILLING MISSING VALUES) for train_hospital_A data ONLY, in order to avoid potential data leakage.

We will save hospital A medians and use them to fill hospital B gaps to prevent data leakage and simulate real world conditions

Since EtC02 column is completely filled with Nan we will just drop it

In [5]:
import json

# Load the training data
print("Loading training data...")
df_train = pd.read_parquet('../data/processed/train_hospital_A.parquet')

# Drop EtCO2 column
df_train = df_train.dropna(axis=1, how='all')  # Drop columns that are all NaN 

# Defining our columns of interest
ignore_cols = ['Patient_ID', 'ICU_Hour', 'SepsisLabel', 'ICULOS']
feature_cols = [col for col in df_train.columns if col not in ignore_cols]

# Calculate the global medians
print("Calculating global medians...")
global_medians = df_train[feature_cols].median()

# Save the medians so we can use them for Hospital B later
global_medians.to_json('../data/cleaned/hospital_A_medians.json', orient='index')
print("Saved medians to hospital_A_medians.json")

# Forward-Fill missing values (Patient by Patient)
print("Forward-filling missing values patient by patient...")
df_train[feature_cols] = df_train.groupby('Patient_ID')[feature_cols].ffill()

# Global Median Fill for any remaining missing values
print("Filling remaining missing values with global medians...")
df_train[feature_cols] = df_train[feature_cols].fillna(global_medians)

# Save the cleaned training data
print("Saving cleaned training data...")
out_path = '../data/cleaned/train_hospital_A_clean.parquet'
df_train.to_parquet(out_path, index=False)

# Verify no missing data is left
missing_left = df_train.isnull().sum().sum()
print(f"Done! Total missing values left: {missing_left}")

Loading training data...
Calculating global medians...
Saved medians to hospital_A_medians.json
Forward-filling missing values patient by patient...
Filling remaining missing values with global medians...
Saving cleaned training data...
Done! Total missing values left: 0


In [2]:
import os
import glob
import pandas as pd
import json
import numpy as np

FEATURE ENGINEERING

In [6]:
# Load clean data
print("Loading cleaned training data...")
df_clean = pd.read_parquet('../data/cleaned/train_hospital_A_clean.parquet')

# Load raw data for test flags
print("Loading raw training data...")
df_raw = pd.read_parquet('../data/processed/train_hospital_A.parquet')

# ICULOS is no longer needed after we added ICU_Hour, so we can drop it
df_clean = df_clean.drop(columns=['ICULOS'])

# Defining the vitals we want to track over time
vitals = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'Resp']

# Creating Test Flags (when doctors tested Lactate, WBC and Bilirubin)
print("Engineering test flags...")
df_clean['Lactate_Test_Flag'] = df_raw['Lactate'].notnull().astype(int)
df_clean['WBC_Test_Flag'] = df_raw['WBC'].notnull().astype(int)
df_clean['Bilirubin_Test_Flag'] = df_raw['Bilirubin_total'].notnull().astype(int)

# Counting the number of tests done
print("Engineering test counts...")
df_clean['Lactate_Test_Count'] = df_clean.groupby('Patient_ID')['Lactate_Test_Flag'].cumsum()
df_clean['WBC_Test_Count'] = df_clean.groupby('Patient_ID')['WBC_Test_Flag'].cumsum()
df_clean['Bilirubin_Test_Count'] = df_clean.groupby('Patient_ID')['Bilirubin_Test_Flag'].cumsum()

# Drop the original test flags as they are no longer needed
df_clean = df_clean.drop(columns=['Lactate_Test_Flag', 'WBC_Test_Flag', 'Bilirubin_Test_Flag'])

# Creating rolling windows and deltas
print("Engineering rolling windows and deltas...")
for col in vitals:
    # 4-Hour Mean
    df_clean[f'{col}_4hr_mean'] = df_clean.groupby('Patient_ID')[col].transform(
        lambda x: x.rolling(window=4, min_periods=1).mean()
    )

    # 4-Hour Standard Deviation
    df_clean[f'{col}_4hr_std'] = df_clean.groupby('Patient_ID')[col].transform(
        lambda x: x.rolling(window=4, min_periods=2).std()
    )

    # 1-Hour Delta
    df_clean[f'{col}_1hr_delta'] = df_clean.groupby('Patient_ID')[col].diff(periods=1)

# Creating Clinical Biomarkers
print("Engineering clinical biomarkers...")

# Shock Index (Heart Rate / Systolic Blood Pressure)
df_clean['Shock_Index'] = df_clean['HR'] / df_clean['SBP']

# BUN to Creatinine Ratio
df_clean['BUN_Creatinine_Ratio'] = df_clean['BUN'] / df_clean['Creatinine']

# System Overload Score (SIRS + qSOFA) 
df_clean['System_Overload_Score'] = (
    (df_clean['HR'] > 100).astype(int) +
    (df_clean['SBP'] < 90).astype(int) +
    (df_clean['Resp'] > 22).astype(int) +
    ((df_clean['Temp'] > 38) | (df_clean['Temp'] < 36)).astype(int)
)

# Cleaning up resulting missing values from the new features
print("Cleaning up missing values from engineered features...")

# The first hour of every patient will have Nan for Delta and Std (we will fill these with 0)
std_cols = [col for col in df_clean.columns if 'std' in col]
delta_cols = [col for col in df_clean.columns if 'delta' in col]
df_clean[std_cols] = df_clean[std_cols].fillna(0)
df_clean[delta_cols] = df_clean[delta_cols].fillna(0)

# Verify no missing data is left
missing_left = df_clean.isnull().sum().sum()
print(f"Done! Total missing values left: {missing_left}")

# Save the engineered dataset
print("Saving engineered dataset...")

out_path = '../data/engineered/train_hospital_A_engineered.parquet'
df_clean.to_parquet(out_path, index=False)

print(f"Feature engineering complete. New dataset shape: {df_clean.shape}")

Loading cleaned training data...
Loading raw training data...
Engineering test flags...
Engineering test counts...
Engineering rolling windows and deltas...
Engineering clinical biomarkers...
Cleaning up missing values from engineered features...
Done! Total missing values left: 0
Saving engineered dataset...
Feature engineering complete. New dataset shape: (790215, 65)


NORMALIZATION

In [7]:
from sklearn.preprocessing import StandardScaler

# Load engineered dataset
print("Loading engineered dataset...")
df_engineered = pd.read_parquet('../data/engineered/train_hospital_A_engineered.parquet')

# Defining excluded columns
excluded_cols = [
    'Patient_ID',
    'ICU_Hour',
    'SepsisLabel',
    'Gender',
    'Unit1',
    'Unit2',
    'System_Overload_Score',
    'Lactate_Test_Count',
    'WBC_Test_Count',
    'Bilirubin_Test_Count'
]

# Applying standard scaling to the features
scale_cols = [col for col in df_engineered.columns if col not in excluded_cols]

print(f"Found {len(scale_cols)} columns to scale.")

scaler = StandardScaler()
df_engineered[scale_cols] = scaler.fit_transform(df_engineered[scale_cols])

print("Normalization complete.")

# Save the normalized dataset
out_path = '../data/final/train_hospital_A_final.parquet'
df_engineered.to_parquet(out_path, index=False)
print(f"Saved normalized dataset. Shape: {df_engineered.shape}")

Loading engineered dataset...
Found 55 columns to scale.
Normalization complete.
Saved normalized dataset. Shape: (790215, 65)
